<a href="https://colab.research.google.com/github/LikhithR916/BIS_Lab/blob/main/Grey%20Wolf%20optimization/1BM24CS147_CloudComputingResourceAllocation_GreyWolfOptimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

# =========================================================================
# 1. SYSTEM SETUP DATA
# =========================================================================
# Tasks: [Length (Million Instructions), Memory Required (GB)]
T_LEN = np.array([500, 1200, 300, 800, 1500, 600, 2000, 400])
T_MEM = np.array([2,   4,    1,   8,   4,    2,   16,   2])

# Virtual Machines (VMs): [Processing Speed (MIPS), Available Memory (GB), Cost/sec ($)]
VM_SPD = np.array([1000, 2000, 4000])
VM_MEM = np.array([4,    8,    16])
VM_CST = np.array([0.05, 0.12, 0.25])

M_TASKS = len(T_LEN)
V_COUNT = len(VM_SPD)

# =========================================================================
# 2. MINIMAL CLOUD SCORING FUNCTION
# =========================================================================
def Calculate_Cloud_Score(plan):
    Time = np.zeros(V_COUNT)
    Cost = 0
    Pen  = 0

    for j, v in enumerate(plan):
        if T_MEM[j] > VM_MEM[v]:
            Pen += 5000
        else:
            t_exec = T_LEN[j] / VM_SPD[v]
            Time[v] += t_exec
            Cost += t_exec * VM_CST[v]

    return (0.6 * np.max(Time)) + (0.4 * Cost) + Pen

# =========================================================================
# 3. SHORT-VARIABLE GREY WOLF OPTIMIZATION (GWO)
# =========================================================================
def grey_wolf_optimization(N=15, max_iter=50):
    # W: Pack matrix (solutions as floats between 0 and VM limit)
    W = np.random.uniform(0, V_COUNT - 1, size=(N, M_TASKS))

    # Initialize leaders
    A, B, D = np.zeros(M_TASKS), np.zeros(M_TASKS), np.zeros(M_TASKS)
    fA, fB, fD = float('inf'), float('inf'), float('inf')

    for t in range(max_iter):
        # --- Step 1: Grade Solutions & Update Leaders ---
        for i in range(N):
            discrete_plan = np.clip(np.round(W[i]), 0, V_COUNT - 1).astype(int)
            score = Calculate_Cloud_Score(discrete_plan)

            if score < fA:
                fD, D = fB, B.copy()
                fB, B = fA, A.copy()
                fA, A = score, W[i].copy()
            elif score < fB:
                fD, D = fB, B.copy()
                fB, B = score, W[i].copy()
            elif score < fD:
                fD, D = score, W[i].copy()

        # --- Step 2: Linear Parameter Decay ---
        a = 2 - t * (2 / max_iter)

        # --- Step 3: Move Pack Towards Top 3 Leaders ---
        for i in range(N):
            for j in range(M_TASKS):
                # Path coordinates pulling toward A, B, and D vectors
                X1 = A[j] - (2 * a * np.random.rand() - a) * abs(2 * np.random.rand() * A[j] - W[i, j])
                X2 = B[j] - (2 * a * np.random.rand() - a) * abs(2 * np.random.rand() * B[j] - W[i, j])
                X3 = D[j] - (2 * a * np.random.rand() - a) * abs(2 * np.random.rand() * D[j] - W[i, j])

                # Update matrix and bound check
                W[i, j] = np.clip((X1 + X2 + X3) / 3, 0, V_COUNT - 1)

    Final_Allocation = np.clip(np.round(A), 0, V_COUNT - 1).astype(int)
    return Final_Allocation, fA

# =========================================================================
# 4. RUN EXECUTION
# =========================================================================
best_plan, optimal_score = grey_wolf_optimization()

print("==================================================")
print("        GWO CLOUD TASK ALLOCATION PLAN            ")
print("==================================================")
print(f"Optimal Allocation Mapping Array: {best_plan}\n")

for task_id, vm_id in enumerate(best_plan):
    print(f"Task {task_id} (Req: {T_MEM[task_id]}GB) -> Assigned to VM {vm_id} (Limit: {VM_MEM[vm_id]}GB, Speed: {VM_SPD[vm_id]} MIPS)")

print(f"\nFinal Optimized System Cost Metric: {optimal_score:.4f}")


        GWO CLOUD TASK ALLOCATION PLAN            
Optimal Allocation Mapping Array: [1 1 1 2 2 0 2 0]

Task 0 (Req: 2GB) -> Assigned to VM 1 (Limit: 8GB, Speed: 2000 MIPS)
Task 1 (Req: 4GB) -> Assigned to VM 1 (Limit: 8GB, Speed: 2000 MIPS)
Task 2 (Req: 1GB) -> Assigned to VM 1 (Limit: 8GB, Speed: 2000 MIPS)
Task 3 (Req: 8GB) -> Assigned to VM 2 (Limit: 16GB, Speed: 4000 MIPS)
Task 4 (Req: 4GB) -> Assigned to VM 2 (Limit: 16GB, Speed: 4000 MIPS)
Task 5 (Req: 2GB) -> Assigned to VM 0 (Limit: 4GB, Speed: 1000 MIPS)
Task 6 (Req: 16GB) -> Assigned to VM 2 (Limit: 16GB, Speed: 4000 MIPS)
Task 7 (Req: 2GB) -> Assigned to VM 0 (Limit: 4GB, Speed: 1000 MIPS)

Final Optimized System Cost Metric: 0.8205
